# MoM 3D Plate - Tetrahedral Element Operations

## Import Packages

In [2]:
using LinearAlgebra  # provides linear algebra toolas 
using StaticArrays   # provides statically typed arrays 
using HCubature      # provides adaptive numerical integration 
using ForwardDiff    # provides forward automatic differentiation 

using Test           # provides testing functionality 

using Plots          # provides plotting functionality 

function hcubature_count(f, a, b; kws...)
    count = 0
    i = hcubature(a, b; kws...) do x
        count += 1
        # display(count)
        f(x)
    end
    return (i..., count)
end;

const Point2D = SVector{2,Float64};
const Point3D = SVector{3,Float64};

const myPoint2D = SVector{2,Real}; # more generic to allow for dual numbders to be created by ForwardDiff
const myPoint3D = SVector{3,Real};

include("mom_struct_definitions.jl");

## Section 1: Two Element (Tetrahedra) Test Case

Assume $P_{\alpha}$ ($P_{\beta}$) to be a tetrahedron bounded by 4 triangular facets $F_{\alpha} \in P_{\alpha}$ ($F_{\beta} \in P_{\beta}$). We wish to compute 

$$
I = \int_{P_{\alpha}} \int_{P_{\beta}} \frac{1}{\| {\mathbf r} - {\mathbf r}' \|} \, 
d\Omega' \, d\Omega \, . 
$$

Given that 

$$
\nabla' \cdot \frac{\mathbf{r} - \mathbf{r}'}{\| \mathbf{r} - \mathbf{r}' \|} = \frac{-2}{\| \mathbf{r} - \mathbf{r}' \|}
$$ 

we obtain that 

$$
I = - \frac{1}{2} \int_{P_{\alpha}} \int_{P_{\beta}} \nabla' \cdot \frac{\mathbf{r} - \mathbf{r}'}{\| \mathbf{r} - \mathbf{r}' \|} d\Omega' \, d\Omega = - \frac{1}{2} \int_{P_{\alpha}} \int_{F_{\beta} \in P_{\beta}} \frac{(\mathbf{r} - \mathbf{r}') \cdot \mathbf{n}'}{\| \mathbf{r} - \mathbf{r}' \|} dS' \, d\Omega
$$

We interchange the order of integration and apply Euler Integration Theorem on both terms seperately to obtain 

In [3]:
refp1 = Point3D(0.,0.,0.); refp2 = Point3D(1.,0.,0.);
refp3 = Point3D(0.,1.,0.); refp4 = Point3D(0.,0.,1.);

refvol  = evalVol(refp1,refp2,refp3,refp4)
refEmat = genBasis(refp1,refp2,refp3,refp4)
indices = Vector(1:12)
refElement = Elem3DLin(refp1,refp2,refp3,refp4,indices,refEmat,refvol)

shift = Point3D(5.,0.,0.)
primep1 = Point3D(0.,0.,0.)+shift; primep2 = Point3D(1.,0.,0.) + shift; 
primep3 = Point3D(0.,1.,0.)+shift; primep4 = Point3D(0.,0.,1.) + shift;

primevol  = evalVol(primep1,primep2,primep3,primep4)
primeEmat = genBasis(primep1,primep2,primep3,primep4)
primeElement = Elem3DLin(primep1,primep2,primep3,primep4,indices,primeEmat,primevol)

Elem3DLin([5.0, 0.0, 0.0], [6.0, 0.0, 0.0], [5.0, 1.0, 0.0], [5.0, 0.0, 1.0], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], [-0.9999999999999996 0.9999999999999996 0.0 0.0; -1.0 0.0 1.0 0.0; -1.0 0.0 0.0 1.0; 5.999999999999997 -4.999999999999997 0.0 0.0], 0.16666666666666666)

In [4]:
integrand = x->1. 
integrate_tet(integrand,refElement)

0.16666666666666663

In [5]:

# define inner integral by integrating over the source domain 
# thus a function of distination variables remains 
inner_integral(rd) = integrate_tet(rs->kernel(rd,rs), primeElement)

outer_integral = integrate_tet(rd->inner_integral(rd), refElement)

0.005555531094601571

In [6]:
inner_integral(Point3D(0.,0.,0.))

0.031667830379712505

## Section 2: Elementary Functions 

In [7]:
refp1 = Point3D(0.,0.,0.); refp2 = Point3D(1.,0.,0.);
refp3 = Point3D(0.,1.,0.); refp4 = Point3D(0.,0.,1.);

refvol  = evalVol(refp1,refp2,refp3,refp4)
refEmat = genBasis(refp1,refp2,refp3,refp4)
indices = Vector(1:12)
refElement = Elem3DLin(refp1,refp2,refp3,refp4,indices,refEmat,refvol)

shift = Point3D(5.,0.,0.)
primep1 = Point3D(0.,0.,0.)+shift; primep2 = Point3D(1.,0.,0.) + shift; 
primep3 = Point3D(0.,1.,0.)+shift; primep4 = Point3D(0.,0.,1.) + shift;

primevol  = evalVol(primep1,primep2,primep3,primep4)
primeEmat = genBasis(primep1,primep2,primep3,primep4)
primeElement = Elem3DLin(primep1,primep2,primep3,primep4,indices,primeEmat,primevol)

Elem3DLin([5.0, 0.0, 0.0], [6.0, 0.0, 0.0], [5.0, 1.0, 0.0], [5.0, 0.0, 1.0], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], [-0.9999999999999996 0.9999999999999996 0.0 0.0; -1.0 0.0 1.0 0.0; -1.0 0.0 0.0 1.0; 5.999999999999997 -4.999999999999997 0.0 0.0], 0.16666666666666666)

In [9]:
points = [refp1, refp2, refp3, refp4, primep1, primep2, primep3, primep4]
edges = [[1,2], [2,3], [3,4], [4,1]]
faces = [[1,2,3], [2,3,4], [3,4,1],[4,1,2],[5,6,7]]

5-element Vector{Vector{Int64}}:
 [1, 2, 3]
 [2, 3, 4]
 [3, 4, 1]
 [4, 1, 2]
 [5, 6, 7]

In [11]:
faces[1]

3-element Vector{Int64}:
 1
 2
 3

In [8]:
evalBasis(primep4,primeElement)

4-element SVector{4, Float64} with indices SOneTo(4):
 -8.881784197001252e-16
  8.881784197001252e-16
  0.0
  1.0

In [15]:
v = vcat(refElement.Emat[:,1],refElement.Emat[:,2],refElement.Emat[:,3])

vprime = vcat(primeElement.Emat[:,1],primeElement.Emat[:,2],primeElement.Emat[:,3])

v*Transpose(vprime)

12×12 MMatrix{12, 12, Float64, 144} with indices SOneTo(12)×SOneTo(12):
  1.0   1.0   1.0  -6.0  -1.0  -0.0  -0.0   5.0  -0.0  -1.0  -0.0  -0.0
  1.0   1.0   1.0  -6.0  -1.0  -0.0  -0.0   5.0  -0.0  -1.0  -0.0  -0.0
  1.0   1.0   1.0  -6.0  -1.0  -0.0  -0.0   5.0  -0.0  -1.0  -0.0  -0.0
 -1.0  -1.0  -1.0   6.0   1.0   0.0   0.0  -5.0   0.0   1.0   0.0   0.0
 -1.0  -1.0  -1.0   6.0   1.0   0.0   0.0  -5.0   0.0   1.0   0.0   0.0
 -0.0  -0.0  -0.0   0.0   0.0   0.0   0.0  -0.0   0.0   0.0   0.0   0.0
 -0.0  -0.0  -0.0   0.0   0.0   0.0   0.0  -0.0   0.0   0.0   0.0   0.0
 -0.0  -0.0  -0.0   0.0   0.0   0.0   0.0  -0.0   0.0   0.0   0.0   0.0
 -0.0  -0.0  -0.0   0.0   0.0   0.0   0.0  -0.0   0.0   0.0   0.0   0.0
 -1.0  -1.0  -1.0   6.0   1.0   0.0   0.0  -5.0   0.0   1.0   0.0   0.0
 -0.0  -0.0  -0.0   0.0   0.0   0.0   0.0  -0.0   0.0   0.0   0.0   0.0
 -0.0  -0.0  -0.0   0.0   0.0   0.0   0.0  -0.0   0.0   0.0   0.0   0.0

In [6]:
#@code_warntype genLocMassMat(refElement)

## Section 3: Integrate over Source Domain, Differentiate w.r.t. and Integrate Again 

**Part-(1/3)** We compute the scalar function 

$$ 
I_{inner}({\mathbf r}) = \int_{P_{\beta}} \frac{\phi_1({\mathbf r})}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, . 
$$ 

In [24]:
basis_fct_beta(rs)      = evalBasis(rs,primeElement)[1]
inv_distance(rd, rs)    = 1/norm(rd - rs)
inner_integrand(rd, rs) = inv_distance(rd, rs)*basis_fct_beta(rs)
inner_integral(rd)      = hcubature(rs->inner_integrand(rd,myPoint3D(rs[1],rs[2],rs[3])), (5.,5.,5.), (6.,6.,6.))[1]

inner_integral (generic function with 1 method)

In [25]:
rd = myPoint3D(.5,.5,.5)
inner_integral(rd)

-1.2105114429821546

**Part-(2/3)** We compute the vector function 

$$
\text{grad}_{\mathbf{r}} I_{inner}({\mathbf r}) = \nabla_{\mathbf{r}} I_{inner}({\mathbf r}) = \nabla_{\mathbf{r}} \int_{P_{\beta}} \frac{\phi_1({\mathbf r})}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, .
$$


In [26]:
dinner_integral(rd) = ForwardDiff.gradient(rd->inner_integral(myPoint3D(rd[1],rd[2],rd[3])), rd)

dinner_integral (generic function with 1 method)

In [27]:
dinner_integral(rd)

3-element SVector{3, Float64} with indices SOneTo(3):
 -0.08057267789194224
 -0.08057267789194222
 -0.08057267789194224

**Part-(3/3)** We compute the scalar  

$$
\int_{P_{\alpha}} \text{grad}_{\mathbf{r}} I_{inner}({\mathbf r}) \, d\Omega = \int_{P_{\alpha}} \nabla_{\mathbf{r}} I_{inner}({\mathbf r}) \, d\Omega = \int_{P_{\alpha}} \nabla_{\mathbf{r}} \int_{P_{\beta}} \frac{\phi_1({\mathbf r})}{\|\mathbf{r}' - \mathbf{r} \|} \, d\Omega' \, d\Omega \, .
$$

In [19]:
outer_integral = hcubature(rd->dinner_integral(myPoint3D(rd[1],rd[2],rd[3])), (0,0,0), (1,1,1))[1]


3-element SVector{3, Float64} with indices SOneTo(3):
 -0.08057337332964216
 -0.08057337332968717
 -0.08057337332995872

All of above in one go. 

In [31]:
a_alphabeta = zeros(4)
b_alphabeta = zeros(4)
c_alphabeta = zeros(4)

for ind = 1:4 
    
    basis_fct_prime(rs)     = evalBasis(rs,primeElement)[ind]
    inv_distance(rd, rs)    = 1/norm(rd - rs)
    inner_integrand(rd, rs) = inv_distance(rd, rs)*basis_fct_prime(rs)
    inner_integral(rd)      = hcubature(rs->inner_integrand(rd,myPoint3D(rs[1],rs[2],rs[3])), (5.,5.,5.), (6.,6.,6.))[1]

    dinner_integral(rd) = ForwardDiff.gradient(rd->inner_integral(myPoint3D(rd[1],rd[2],rd[3])), rd)

    outer_integral = hcubature(rd->dinner_integral(myPoint3D(rd[1],rd[2],rd[3])), (0,0,0), (1,1,1))[1]
 
    a_alphabeta[ind], b_alphabeta[ind], c_alphabeta[ind] = outer_integral
    
end 

In [33]:
a_alpha = refEmat[1,:]; b_alpha = refEmat[2,:]; c_alpha = refEmat[3,:]  

4-element SVector{4, Float64} with indices SOneTo(4):
 -1.0
  0.0
  0.0
  1.0